In [ ]:
"""
MaxViT Training Script for Ultrasound Image Classification
Classes: normal, benign, malignant
"""

import os
import time
import copy
import json
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import maxvit_t, MaxVit_T_Weights

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ─────────────────────────────────────────────────────────────
#   VARIABLES
# ─────────────────────────────────────────────────────────────

DATA_DIR       = "./unified_ultrasound_dataset"
OUTPUT_DIR     = "outputs"

NUM_CLASSES    = 3
IMG_SIZE       = 224
BATCH_SIZE     = 16
NUM_EPOCHS     = 30
LR             = 2e-4        # initial learning rate (head only)
WEIGHT_DECAY   = 1e-4
UNFREEZE_EPOCH = 5           # epoch at which backbone is unfrozen for full fine-tuning
DROPOUT        = 0.3
NUM_WORKERS    = 4
SEED           = 42
AMP            = True        # mixed precision (requires CUDA)

# ─────────────────────────────────────────────────────────────


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True


# ── Data ──────────────────────────────────────────────────────

def get_transforms():
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


def get_dataloaders():
    train_tf, val_tf = get_transforms()

    train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"),   val_tf)
    test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"),  val_tf)

    # Weighted sampler to handle class imbalance
    class_counts   = np.bincount(train_ds.targets)
    class_weights  = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    sample_weights = class_weights[train_ds.targets]
    sampler = torch.utils.data.WeightedRandomSampler(
        sample_weights, len(sample_weights), replacement=True
    )

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

    class_names = [k for k, _ in sorted(train_ds.class_to_idx.items(), key=lambda x: x[1])]

    print(f"\nDataset sizes  ->  train: {len(train_ds)}  |  val: {len(val_ds)}  |  test: {len(test_ds)}")
    print(f"Class mapping  ->  {train_ds.class_to_idx}")
    print(f"Class counts   ->  {dict(zip(train_ds.classes, class_counts.tolist()))}\n")

    return train_dl, val_dl, test_dl, class_names


# ── Model ─────────────────────────────────────────────────────

def build_model(freeze_backbone=True):
    model = maxvit_t(weights=MaxVit_T_Weights.DEFAULT)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Dynamically find the last Linear layer and replace it
    layers = list(model.classifier.children())

    last_linear_idx = None
    for i, layer in enumerate(layers):
        if isinstance(layer, nn.Linear):
            last_linear_idx = i

    if last_linear_idx is None:
        raise RuntimeError(f"No Linear found in classifier: {model.classifier}")

    in_features = layers[last_linear_idx].in_features
    print(f"Replacing classifier[{last_linear_idx}] Linear({in_features} -> 1000) "
          f"with Linear({in_features} -> {NUM_CLASSES})")

    # Replace the final Linear
    layers[last_linear_idx] = nn.Linear(in_features, NUM_CLASSES)

    # Insert a Dropout before it if one isn't already there
    if not isinstance(layers[last_linear_idx - 1], nn.Dropout):
        layers.insert(last_linear_idx, nn.Dropout(p=DROPOUT))

    model.classifier = nn.Sequential(*layers)

    # Head is always trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model


def unfreeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = True
    print("Backbone unfrozen — full fine-tuning enabled.")


# ── Training loop ─────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.autocast(device_type=device.type, enabled=scaler is not None):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += imgs.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.autocast(device_type=device.type, enabled=True):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        running_loss += loss.item() * imgs.size(0)
        preds         = outputs.argmax(1)
        correct      += (preds == labels).sum().item()
        total        += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels


# ── Plots ─────────────────────────────────────────────────────

def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

    axes[1].plot(epochs, history["train_acc"], label="Train")
    axes[1].plot(epochs, history["val_acc"],   label="Val")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "training_curves.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")


def plot_confusion_matrix(labels, preds, class_names):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title("Confusion Matrix - Test Set")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")


# ── Main ──────────────────────────────────────────────────────

def main():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_dl, val_dl, test_dl, class_names = get_dataloaders()

    model     = build_model(freeze_backbone=True).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda") if (AMP and device.type == "cuda") else None

    history      = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = None

    for epoch in range(1, NUM_EPOCHS + 1):

        # Unfreeze backbone after warm-up phase
        if epoch == UNFREEZE_EPOCH:
            unfreeze_backbone(model)
            optimizer = optim.AdamW(model.parameters(),
                                    lr=LR * 0.1, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer,
                                          T_max=NUM_EPOCHS - epoch, eta_min=1e-6)

        t0 = time.time()
        tr_loss, tr_acc         = train_one_epoch(model, train_dl, criterion, optimizer, scaler, device)
        vl_loss, vl_acc, _, _   = evaluate(model, val_dl, criterion, device)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)

        print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
              f"| Train loss: {tr_loss:.4f}  acc: {tr_acc:.4f} "
              f"| Val loss: {vl_loss:.4f}  acc: {vl_acc:.4f} "
              f"| {time.time() - t0:.1f}s")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, os.path.join(OUTPUT_DIR, "best_model.pth"))
            print(f"  New best val acc: {best_val_acc:.4f} — model saved.")

    # ── Test set evaluation ───────────────────────────────────
    model.load_state_dict(best_weights)
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_dl, criterion, device)

    print(f"\n{'='*55}")
    print(f"Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.4f}")
    print(f"{'='*55}\n")
    print(classification_report(test_labels, test_preds, target_names=class_names))

    plot_history(history)
    plot_confusion_matrix(test_labels, test_preds, class_names)

    results = {
        "best_val_acc": best_val_acc,
        "test_acc":     test_acc,
        "test_loss":    test_loss,
        "class_names":  class_names,
        "num_epochs":   NUM_EPOCHS,
        "batch_size":   BATCH_SIZE,
        "lr":           LR,
    }
    with open(os.path.join(OUTPUT_DIR, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    print(f"\nAll outputs saved to: {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

Device: cuda

Dataset sizes  ->  train: 1648  |  val: 211  |  test: 415
Class mapping  ->  {'benign': 0, 'malignant': 1, 'normal': 2}
Class counts   ->  {'benign': 702, 'malignant': 546, 'normal': 400}

Replacing classifier[5] Linear(512 -> 1000) with Linear(512 -> 3)
Epoch [01/30] | Train loss: 0.9540  acc: 0.5485 | Val loss: 0.9470  acc: 0.6066 | 16.0s
  New best val acc: 0.6066 — model saved.
Epoch [02/30] | Train loss: 0.8523  acc: 0.6578 | Val loss: 0.8965  acc: 0.5877 | 13.1s
Epoch [03/30] | Train loss: 0.8430  acc: 0.6450 | Val loss: 0.8849  acc: 0.6114 | 13.1s
  New best val acc: 0.6114 — model saved.
Epoch [04/30] | Train loss: 0.7748  acc: 0.6996 | Val loss: 0.8730  acc: 0.6114 | 13.1s
Backbone unfrozen — full fine-tuning enabled.
Epoch [05/30] | Train loss: 0.7742  acc: 0.6984 | Val loss: 0.8151  acc: 0.6872 | 27.9s
  New best val acc: 0.6872 — model saved.
Epoch [06/30] | Train loss: 0.7564  acc: 0.7100 | Val loss: 0.7712  acc: 0.7109 | 23.0s
  New best val acc: 0.7109 — mo